# Day 086 | SPT to Shear Wave Velocity (ANN + Ensemble)

**Author:** Ripon Chandra Malo

## Capstone Project: Predicting Shear Wave Velocity from SPT Data

Shear wave velocity (Vs) is a critical parameter for seismic site characterization, but direct measurement through geophysical tests is expensive. Standard Penetration Test (SPT) data is widely available and more economical. In this capstone project, we build ANN (PyTorch) and ensemble models (XGBoost, Random Forest) to predict Vs from SPT blow counts and other geotechnical parameters.

**Objectives:**
- Generate realistic synthetic SPT-Vs dataset (250 rows)
- Build and train an ANN using PyTorch
- Train XGBoost and Random Forest ensemble models
- Compare all models with comprehensive evaluation metrics
- Visualize predictions and feature importance

In [ ]:
# ───── Part 1: Imports and Setup ─────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed. Install with: pip install xgboost')

np.random.seed(42)
torch.manual_seed(42)
print('All libraries loaded successfully.')

## Part 2: Synthetic Data Generation

We generate a realistic dataset based on established empirical correlations between SPT-N values and Vs. The well-known relationships include:
- **Imai & Tonouchi (1982):** Vs = 97 * N^0.314
- **Ohta & Goto (1978):** Vs depends on N, depth, geological age, and soil type

Our synthetic data incorporates depth, soil type, fines content, groundwater depth, and effective stress as additional features.

In [ ]:
# ───── Part 2: Generate Synthetic SPT-Vs Dataset ─────
n_samples = 250

# Depth (m) - typical SPT investigation range
depth = np.random.uniform(1.5, 30.0, n_samples)

# Soil type: 0=Clay, 1=Silt, 2=Sand, 3=Gravel
soil_type = np.random.choice([0, 1, 2, 3], n_samples, p=[0.25, 0.20, 0.35, 0.20])

# Fines content (%)
fines_content = np.where(soil_type == 0, np.random.uniform(60, 100, n_samples),
                np.where(soil_type == 1, np.random.uniform(30, 70, n_samples),
                np.where(soil_type == 2, np.random.uniform(0, 20, n_samples),
                         np.random.uniform(0, 10, n_samples))))

# Groundwater table depth (m)
gwt_depth = np.random.uniform(0.5, 10.0, n_samples)

# SPT-N values (correlated with depth and soil type)
base_N = 5 + 1.2 * depth + soil_type * 4
spt_n = np.clip(base_N + np.random.normal(0, 5, n_samples), 1, 80).astype(int)

# Effective vertical stress (kPa) - simplified
unit_weight = np.where(soil_type <= 1, 17.0, 19.0)  # kN/m3
gamma_w = 9.81
sigma_v_eff = unit_weight * depth - gamma_w * np.maximum(depth - gwt_depth, 0)
sigma_v_eff = np.clip(sigma_v_eff, 5, 600)

# Vs based on enhanced empirical correlation
soil_factor = np.where(soil_type == 0, 0.85,
              np.where(soil_type == 1, 0.95,
              np.where(soil_type == 2, 1.05, 1.15)))

vs = (97 * spt_n**0.314 * soil_factor *
      (1 + 0.003 * depth) *
      (1 - 0.001 * fines_content)) + np.random.normal(0, 15, n_samples)
vs = np.clip(vs, 60, 600)

# Create DataFrame
df = pd.DataFrame({
    'depth_m': np.round(depth, 2),
    'spt_n': spt_n,
    'soil_type': soil_type,
    'fines_content': np.round(fines_content, 1),
    'gwt_depth_m': np.round(gwt_depth, 2),
    'sigma_v_eff_kPa': np.round(sigma_v_eff, 2),
    'vs_m_s': np.round(vs, 1)
})

df.to_csv('spt_vs_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# ───── Part 3: Exploratory Data Analysis ─────
print('=== Dataset Statistics ===')
print(df.describe().round(2))
print(f'\nSoil type distribution:')
soil_labels = {0: 'Clay', 1: 'Silt', 2: 'Sand', 3: 'Gravel'}
for st, label in soil_labels.items():
    count = (df['soil_type'] == st).sum()
    print(f'  {label}: {count} samples ({100*count/len(df):.1f}%)')

In [ ]:
# ───── Part 4: Visualization ─────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (a) SPT-N vs Vs colored by soil type
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
for st, label in soil_labels.items():
    mask = df['soil_type'] == st
    axes[0, 0].scatter(df.loc[mask, 'spt_n'], df.loc[mask, 'vs_m_s'],
                       c=colors[st], label=label, alpha=0.6, edgecolors='k', linewidth=0.3)
axes[0, 0].set_xlabel('SPT-N', fontsize=12)
axes[0, 0].set_ylabel('Vs (m/s)', fontsize=12)
axes[0, 0].set_title('(a) SPT-N vs Vs by Soil Type', fontsize=13)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# (b) Depth vs Vs
sc = axes[0, 1].scatter(df['depth_m'], df['vs_m_s'], c=df['spt_n'],
                        cmap='viridis', alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, ax=axes[0, 1], label='SPT-N')
axes[0, 1].set_xlabel('Depth (m)', fontsize=12)
axes[0, 1].set_ylabel('Vs (m/s)', fontsize=12)
axes[0, 1].set_title('(b) Depth vs Vs (colored by SPT-N)', fontsize=13)
axes[0, 1].grid(True, alpha=0.3)

# (c) Distribution of Vs
axes[1, 0].hist(df['vs_m_s'], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(df['vs_m_s'].mean(), color='red', linestyle='--',
                    label=f'Mean={df["vs_m_s"].mean():.0f} m/s')
axes[1, 0].set_xlabel('Vs (m/s)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('(c) Vs Distribution', fontsize=13)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# (d) Correlation heatmap
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1, 1],
            square=True, linewidths=0.5)
axes[1, 1].set_title('(d) Feature Correlation Matrix', fontsize=13)

plt.tight_layout()
plt.savefig('eda_spt_vs.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')

In [ ]:
# ───── Part 5: Data Preparation ─────
features = ['depth_m', 'spt_n', 'soil_type', 'fines_content', 'gwt_depth_m', 'sigma_v_eff_kPa']
target = 'vs_m_s'

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# Scale features
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_sc = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

## Part 6: ANN Model (PyTorch)

We build a feed-forward neural network with:
- 3 hidden layers (64, 32, 16 neurons)
- ReLU activations and batch normalization
- Dropout for regularization

In [ ]:
# ───── Part 6: Define ANN Architecture ─────
class VsPredictor(nn.Module):
    def __init__(self, input_dim):
        super(VsPredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model_ann = VsPredictor(input_dim=X_train_sc.shape[1])
print(model_ann)
total_params = sum(p.numel() for p in model_ann.parameters())
print(f'\nTotal parameters: {total_params}')

In [ ]:
# ───── Part 7: Train ANN ─────
X_train_t = torch.FloatTensor(X_train_sc)
y_train_t = torch.FloatTensor(y_train_sc).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test_sc)
y_test_t = torch.FloatTensor(y_test_sc).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.MSELoss()
optimizer = optim.Adam(model_ann.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

epochs = 200
train_losses, val_losses = [], []

for epoch in range(epochs):
    model_ann.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model_ann(batch_X)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))

    model_ann.eval()
    with torch.no_grad():
        val_pred = model_ann(X_test_t)
        val_loss = criterion(val_pred, y_test_t).item()
        val_losses.append(val_loss)

    scheduler.step()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] - Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_loss:.4f}')

print('ANN training complete.')

In [ ]:
# ───── Part 8: ANN Training History ─────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Training Loss', color='steelblue', linewidth=1.5)
ax.plot(val_losses, label='Validation Loss', color='coral', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('ANN Training History', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ann_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 9: ANN Predictions ─────
model_ann.eval()
with torch.no_grad():
    y_pred_ann_sc = model_ann(X_test_t).numpy().ravel()

y_pred_ann = scaler_y.inverse_transform(y_pred_ann_sc.reshape(-1, 1)).ravel()

r2_ann = r2_score(y_test, y_pred_ann)
rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))
mae_ann = mean_absolute_error(y_test, y_pred_ann)

print('=== ANN Performance ===')
print(f'R-squared:  {r2_ann:.4f}')
print(f'RMSE: {rmse_ann:.2f} m/s')
print(f'MAE:  {mae_ann:.2f} m/s')

## Part 10: Ensemble Models

We train two popular ensemble methods:
1. **Random Forest** -- bagging-based ensemble of decision trees
2. **XGBoost** -- gradient boosting with regularization

In [ ]:
# ───── Part 10: Random Forest ─────
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print('=== Random Forest Performance ===')
print(f'R-squared:  {r2_rf:.4f}')
print(f'RMSE: {rmse_rf:.2f} m/s')
print(f'MAE:  {mae_rf:.2f} m/s')

In [ ]:
# ───── Part 11: XGBoost ─────
if HAS_XGB:
    xgb_model = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_xgb = xgb_model.predict(X_test)

    r2_xgb = r2_score(y_test, y_pred_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

    print('=== XGBoost Performance ===')
    print(f'R-squared:  {r2_xgb:.4f}')
    print(f'RMSE: {rmse_xgb:.2f} m/s')
    print(f'MAE:  {mae_xgb:.2f} m/s')
else:
    print('XGBoost not available - skipping.')
    y_pred_xgb = None
    r2_xgb = rmse_xgb = mae_xgb = None

In [ ]:
# ───── Part 12: Empirical Correlation Baseline ─────
# Imai & Tonouchi (1982): Vs = 97 * N^0.314
spt_n_test = X_test[:, 1]  # column index 1 = spt_n
y_pred_empirical = 97.0 * spt_n_test**0.314

r2_emp = r2_score(y_test, y_pred_empirical)
rmse_emp = np.sqrt(mean_squared_error(y_test, y_pred_empirical))
mae_emp = mean_absolute_error(y_test, y_pred_empirical)

print('=== Empirical Baseline (Imai & Tonouchi 1982) ===')
print(f'R-squared:  {r2_emp:.4f}')
print(f'RMSE: {rmse_emp:.2f} m/s')
print(f'MAE:  {mae_emp:.2f} m/s')

In [ ]:
# ───── Part 13: Model Comparison Summary ─────
results = pd.DataFrame({
    'Model': ['Empirical (Imai)', 'ANN (PyTorch)', 'Random Forest', 'XGBoost'],
    'R2': [r2_emp, r2_ann, r2_rf, r2_xgb if r2_xgb else np.nan],
    'RMSE (m/s)': [rmse_emp, rmse_ann, rmse_rf, rmse_xgb if rmse_xgb else np.nan],
    'MAE (m/s)': [mae_emp, mae_ann, mae_rf, mae_xgb if mae_xgb else np.nan]
}).dropna()

results = results.sort_values('R2', ascending=False).reset_index(drop=True)
print('=== Model Comparison ===')
print(results.to_string(index=False))

In [ ]:
# ───── Part 14: Comparison Bar Chart ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = results['Model'].tolist()
bar_colors = ['#90CAF9', '#EF5350', '#66BB6A', '#FFA726'][:len(model_names)]

axes[0].barh(model_names, results['R2'], color=bar_colors, edgecolor='black')
axes[0].set_xlabel('R-squared', fontsize=12)
axes[0].set_title('R-squared (higher = better)', fontsize=13)
axes[0].set_xlim(0, 1.05)
for i, v in enumerate(results['R2']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)

axes[1].barh(model_names, results['RMSE (m/s)'], color=bar_colors, edgecolor='black')
axes[1].set_xlabel('RMSE (m/s)', fontsize=12)
axes[1].set_title('RMSE (lower = better)', fontsize=13)
for i, v in enumerate(results['RMSE (m/s)']):
    axes[1].text(v + 0.3, i, f'{v:.1f}', va='center', fontsize=10)

axes[2].barh(model_names, results['MAE (m/s)'], color=bar_colors, edgecolor='black')
axes[2].set_xlabel('MAE (m/s)', fontsize=12)
axes[2].set_title('MAE (lower = better)', fontsize=13)
for i, v in enumerate(results['MAE (m/s)']):
    axes[2].text(v + 0.3, i, f'{v:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 15: Predicted vs Observed Plots ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmin, vmax = min(y_test.min(), 60), max(y_test.max(), 400)

model_preds = [('ANN', y_pred_ann), ('Random Forest', y_pred_rf)]
if y_pred_xgb is not None:
    model_preds.append(('XGBoost', y_pred_xgb))

for idx, (name, y_pred) in enumerate(model_preds):
    axes[idx].scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidth=0.3, c='steelblue')
    axes[idx].plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=2, label='1:1 Line')
    axes[idx].set_xlabel('Observed Vs (m/s)', fontsize=11)
    axes[idx].set_ylabel('Predicted Vs (m/s)', fontsize=11)
    r2 = r2_score(y_test, y_pred)
    axes[idx].set_title(f'{name} (R2={r2:.3f})', fontsize=13)
    axes[idx].set_xlim(vmin, vmax)
    axes[idx].set_ylim(vmin, vmax)
    axes[idx].set_aspect('equal')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pred_vs_obs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 16: Feature Importance ─────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rf_imp = pd.Series(rf_model.feature_importances_, index=features).sort_values()
rf_imp.plot(kind='barh', ax=axes[0], color='forestgreen', edgecolor='black')
axes[0].set_title('Random Forest Feature Importance', fontsize=13)
axes[0].set_xlabel('Importance', fontsize=11)

if HAS_XGB and y_pred_xgb is not None:
    xgb_imp = pd.Series(xgb_model.feature_importances_, index=features).sort_values()
    xgb_imp.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='black')
    axes[1].set_title('XGBoost Feature Importance', fontsize=13)
    axes[1].set_xlabel('Importance', fontsize=11)
else:
    axes[1].text(0.5, 0.5, 'XGBoost not available', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=14)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 17: Residual Analysis ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, y_pred) in enumerate(model_preds):
    residuals = y_test - y_pred
    axes[idx].scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidth=0.3, c='steelblue')
    axes[idx].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[idx].set_xlabel('Predicted Vs (m/s)', fontsize=11)
    axes[idx].set_ylabel('Residual (m/s)', fontsize=11)
    axes[idx].set_title(f'{name} Residuals', fontsize=13)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 18: Vs Profile with Depth ─────
test_df = pd.DataFrame(X_test, columns=features)
test_df['vs_obs'] = y_test
test_df['vs_ann'] = y_pred_ann
test_df['vs_rf'] = y_pred_rf
if y_pred_xgb is not None:
    test_df['vs_xgb'] = y_pred_xgb

test_sorted = test_df.sort_values('depth_m')

fig, ax = plt.subplots(figsize=(8, 10))
ax.plot(test_sorted['vs_obs'], test_sorted['depth_m'], 'ko', markersize=5, label='Observed', zorder=5)
ax.plot(test_sorted['vs_ann'], test_sorted['depth_m'], 'b^', markersize=4, alpha=0.7, label='ANN')
ax.plot(test_sorted['vs_rf'], test_sorted['depth_m'], 'gs', markersize=4, alpha=0.7, label='RF')
if y_pred_xgb is not None:
    ax.plot(test_sorted['vs_xgb'], test_sorted['depth_m'], 'r+', markersize=5, alpha=0.7, label='XGBoost')
ax.invert_yaxis()
ax.set_xlabel('Vs (m/s)', fontsize=12)
ax.set_ylabel('Depth (m)', fontsize=12)
ax.set_title('Vs Profile with Depth - Model Predictions vs Observed', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vs_depth_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **ML models significantly outperform traditional empirical correlations** for predicting Vs from SPT data when additional features (depth, soil type, fines content) are available.

2. **SPT-N and depth are the most important features** for Vs prediction, consistent with established geophysical understanding.

3. **Ensemble methods (XGBoost, Random Forest) are competitive with ANNs** and often easier to train and interpret for tabular geotechnical data.

4. **Multi-feature models capture soil-type-dependent behavior** that single-variable empirical correlations miss entirely.

5. **Residual analysis shows approximately uniform error distribution**, indicating no systematic bias in the ML predictions.

## Next Steps

- Apply models to real SPT-Vs datasets from specific geological settings
- Incorporate geological age as an additional feature
- Use Bayesian optimization for hyperparameter tuning
- Deploy as a web application for field engineers
- Extend to predict Vs30 for seismic site classification